In [68]:
import os
import json
from datetime import datetime, timezone

In [69]:
location = ""

In [70]:
def load_json(filename):    #Twitter json file loader

    file_path = os.path.join(location, filename)    #OS library used to make the code OS independent
    
    with open(file_path, "r") as file:
        content = file.read().strip()
        content = content.split("=", 1)[1].strip()    #To get content after = without whitespaces or newlines

    return json.loads(content)

In [71]:
def format_date(date_str):    #Function to get a proper date / time format
    
    date_part, time_part = date_str.split("T")    #To split the string into a separate date and time
    time_part = time_part.rstrip("Z")    #To remove the trailing "Z"
    time_part = time_part.split(".")[0]    #To remove milliseconds 
    year, month, day = date_part.split("-")    #To split the date into its constituents
    formatted_date = f"{day}/{month}/{year} {time_part}"    #Format - DD/MM/YYYY HH:MM:SS

    return formatted_date

In [72]:
def tweet_id_to_datetime(tweet_id):    #Function to convert Tweet ID to datetime
    
    twitter_epoch = 1288834974657    #Twitter epoch(January 1, 2010, 00:00:00 UTC)
    timestamp = (int(tweet_id) >> 22) + twitter_epoch    #Extracting timestamp from the tweet ID
    tweet_datetime = datetime.fromtimestamp(timestamp / 1000.0, tz=timezone.utc)    #Converting timestamp to a timezone-aware datetime object
    
    return tweet_datetime

In [ ]:
def get_account_details():

    account_data = load_json("account.js")
    account_info = account_data[0].get("account", {}) if account_data else {}

    return {"E - mail": account_info.get("email", "Unknown"),
            "Username": account_info.get("username", "Unknown"),
            "Created at": format_date(account_info.get("createdAt", "Unknown")),
            "Account ID": account_info.get("accountId", "Unknown"),
            "Display Name": account_info.get("accountDisplayName", "Unknown")}

def get_interests_and_demographics():

    personalization_data = load_json("personalization.js")
    data = personalization_data[0].get("p13nData", {})
    languages = [i.get("language", "Unknown") for i in data.get("demographics", {}).get("languages", []) if (i.get("isDisabled") == False and i.get("language") != "No linguistic content")]
    gender = data.get("demographics", {}).get("genderInfo", {}).get("gender", "Unknown")
    shows = data.get("interests", {}).get("shows", [])
    locations = data.get("locationHistory", [])
    age = data.get("inferredAgeInfo", {}).get("age", "Unknown")
    interests_list = [i.get("name", "Unknown") for i in data.get("interests", {}).get("interests", []) if i.get("name")[0] != "$"] if personalization_data else []
    
    return languages, gender, shows, locations, age, interests_list

languages, gender, shows, locations, age, interests = get_interests_and_demographics()
account_details = get_account_details()
ID = account_details["Account ID"]

print("\n".join([f"{a} - {b}" for a, b in account_details.items()]))    #Unpacking the dictionary into a and b, making a list with them, and joining them to make the output more readable
print("Languages -", ", ".join(languages))
print("Gender -", gender)
print("Shows -", ", ".join(shows))
print("Locations -", ", ".join(locations))
print("Age -", ", ".join(age))
print("Interests -", ", ".join(interests))

In [ ]:
def get_mutual_follows():

    followers_data = load_json("follower.js")
    following_data = load_json("following.js")
    followers_ids = {entry["follower"]["accountId"] for entry in followers_data}
    following_ids = {entry["following"]["accountId"] for entry in following_data}
    mutual_ids = followers_ids.intersection(following_ids)    #To find mutual followers
    
    with open("mutual_follows.txt", "w") as file:    #Saving links to mutual followers in a file
        
        for account_id in mutual_ids:
            file.write(f"https://x.com/intent/user?user_id={account_id}\n")

    return len(mutual_ids)

print("Mutual Followers:", get_mutual_follows())

In [ ]:
def analysis_ads():

    ad_data = load_json("ad-engagements.js")
    unique_advertisers_and_associations = {}
    unique_OSs = set()

    for entry in ad_data:
        ads = entry.get('ad', {}).get("adsUserData", {}).get("adEngagements", {}).get("engagements", [])

        for ad in ads:
            impression = ad.get("impressionAttributes", {})
            advertiser_name = impression.get("advertiserInfo", {}).get("advertiserName")
            targeting_values = [i["targetingValue"] for i in impression.get("matchedTargetingCriteria", [])]    
            unique_advertisers_and_associations[advertiser_name] = targeting_values
            unique_OSs.add(impression.get("deviceInfo", {}).get("osType"))

    return unique_advertisers_and_associations, unique_OSs

data = analysis_ads()

print("Advertisers and their targeting criteria -")

for i in data[0]:
    print(i, "-", ", ".join(data[0][i]))

print("\nOperating Systems used -", ", ".join(data[1]))

In [ ]:
def analyze_tweets():

    tweets = load_json("tweet-headers.js")
    likes = load_json("like.js")
    total_liked_tweets = len(likes)
    tweets_by_day = {}
       
    for tweet in tweets:
        created_at = tweet.get("tweet", {}).get("created_at")
        parsed_date = datetime.strptime(created_at, "%a %b %d %H:%M:%S %z %Y")    #Parsing the date
        tweet_date = parsed_date.strftime("%d/%m/%Y")    #Format: YYYY/MM/DD
        tweets_by_day[tweet_date] = tweets_by_day.get(tweet_date, 0) + 1    #Creates an entry in the dictionary if not present and adds 1 if present

    return {"liked_tweets": total_liked_tweets, 
            "tweets_by_day": tweets_by_day}
    
tweet_analysis = analyze_tweets()

print(f"Total Liked Tweets - {tweet_analysis["liked_tweets"]}\n")
print(f"Tweet Counts by Date -\n{", ".join([f"{a} - {b}" for a, b in tweet_analysis["tweets_by_day"].items()][::-1])}")

In [ ]:
def analyze_direct_messages():

    dm_headers = load_json("direct-message-headers.js")  
    users_and_messages = {}    #Tracking DM contacts and message counts with them

    for conversation in dm_headers:
        dm_conv = conversation.get("dmConversation")
        messages = dm_conv.get("messages", [])
        participants = dm_conv.get("conversationId").split("-")

        if len(participants) != 2:    #Not considering group DMs
            continue

        other_participant = [p for p in participants if p != ID][0]    #Identifying the other participant in the conversation
        users_and_messages[other_participant] = len(messages)

    most_messaged_user = max(users_and_messages, key = users_and_messages.get, default = 0)

    return {"Most messaged user": most_messaged_user,
            "Message count": users_and_messages.get(most_messaged_user, 0)}
    
print("\n".join(f"{a} - {b}" for a, b in analyze_direct_messages().items()))